![iceberg-logo](https://www.apache.org/logos/res/iceberg/iceberg.png)

# Icebergの世界を体感しよう

## 本章のねらい：
- **Icebergの操作に慣れよう**  
- **Icebergのコンセプトに触れよう**

# Introduction

[Apache Iceberg](https://iceberg.apache.org/)は、2017年にNetflixが開発した、大規模なデータセットに最適化されたOpen Table Formatの一種です。  
オブジェクトストレージやHDFS上のデータをTrinoやSparkなどのエンジン/ツールで操作する基盤を運用する中で突き当たる機能的、性能的な限界を突破するために生まれました。  

Icebergの代表的な特徴として、以下のような点が挙げられます。
- 同時書き込み、読み込みの一貫性(ACID)
    - [楽観的並行性制御](https://ja.wikipedia.org/wiki/%E6%A5%BD%E8%A6%B3%E7%9A%84%E4%B8%A6%E8%A1%8C%E6%80%A7%E5%88%B6%E5%BE%A1)等によりSERIALIZABLE Isolationを実現
- タイムトラベル
    - 過去のある時点の断面を遡って参照できる
- スキーマ/パーティション・エボリューション
    - スキーマやパーティションの変更に柔軟に対応

本ハンズオンでは **PyIceberg** (Pythonクライアント) と **DuckDB** を使って、これらの特徴を実際に体験します。  
Spark等の重いエンジンは不要で、Pythonだけで完結します。

詳細を知りたい方は以下をご参照ください。  
[Iceberg 公式 Doc](https://iceberg.apache.org/docs/latest/)  
[Apache Iceberg とは何か](https://bering.hatenadiary.com/entry/2023/09/24/175953)

# ハンズオン

## 1. セットアップ

PyIcebergのカタログを初期化します。  
ここでは **SQLiteカタログ** を使います。カタログとはIcebergテーブルのメタデータを管理するコンポーネントです。  
SQLiteファイル1つで完結するため、外部サービス(MinIO等)は不要です。

In [ ]:
import shutil
from pyiceberg.catalog.sql import SqlCatalog

WAREHOUSE_PATH = "/home/iceberg/warehouse"

# ノートブックを何度も再実行できるようにするため、既存のwarehouseを削除
shutil.rmtree(WAREHOUSE_PATH, ignore_errors=True)

catalog = SqlCatalog(
    "demo",
    **{
        "uri": f"sqlite:///{WAREHOUSE_PATH}/catalog.db",
        "warehouse": f"file://{WAREHOUSE_PATH}",
    },
)

catalog.create_namespace("nyc")
print("カタログの初期化完了")

## 2. サンプルデータを作ろう

本ハンズオンではニューヨーク市のタクシー移動データを模したサンプルデータを使用します。  
大きなファイルのダウンロードは不要で、Pythonでその場生成します。

In [ ]:
import pandas as pd
import pyarrow as pa
from datetime import datetime, timedelta
import random

random.seed(42)

n = 1000
base_time = datetime(2021, 4, 1)

df = pd.DataFrame({
    "vendor_id":        [random.choice([1, 2]) for _ in range(n)],
    "pickup_datetime":  [base_time + timedelta(minutes=i*2) for i in range(n)],
    "dropoff_datetime": [base_time + timedelta(minutes=i*2+random.randint(5,30)) for i in range(n)],
    "passenger_count":  [random.randint(1, 4) for _ in range(n)],
    "trip_distance":    [round(random.uniform(0.5, 20.0), 2) for _ in range(n)],
    "fare_amount":      [round(random.uniform(3.0, 60.0), 2) for _ in range(n)],
})

print(f"レコード数: {len(df)}")
df.head()

## 3. Icebergテーブルを作ってみよう

サンプルデータをIcebergテーブルとして保存します。

In [ ]:
from pyiceberg.schema import Schema
from pyiceberg.types import (
    NestedField, LongType, TimestampType, DoubleType, FloatType
)

schema = Schema(
    NestedField(1,  "vendor_id",       LongType(),      required=False),
    NestedField(2,  "pickup_datetime",  TimestampType(), required=False),
    NestedField(3,  "dropoff_datetime", TimestampType(), required=False),
    NestedField(4,  "passenger_count", LongType(),      required=False),
    NestedField(5,  "trip_distance",   DoubleType(),    required=False),
    NestedField(6,  "fare_amount",     DoubleType(),    required=False),
)

table = catalog.create_table("nyc.taxis", schema=schema)
print(f"テーブル作成完了: {table.name()}")

In [ ]:
arrow_table = pa.Table.from_pandas(df, schema=table.schema().as_arrow())
table.append(arrow_table)

print(f"データ投入完了: {len(table.scan().to_arrow())} レコード")

DuckDBでIcebergテーブルの中身を確認してみましょう。

In [ ]:
import duckdb

conn = duckdb.connect()
conn.execute("INSTALL iceberg; LOAD iceberg;")

# PyIcebergで現在のデータファイルのパスを取得してDuckDBで読み込む
def query_table(tbl, sql="SELECT * FROM tbl LIMIT 5"):
    arrow = tbl.scan().to_arrow()
    return conn.execute(sql.replace("tbl", "arrow")).df()

query_table(table, "SELECT * FROM arrow LIMIT 5")

### Icebergテーブルのストレージ構造を見てみよう

Icebergテーブルはどのようにファイルとして保存されているか確認してみましょう。

In [ ]:
import os

for root, dirs, files in os.walk(WAREHOUSE_PATH):
    level = root.replace(WAREHOUSE_PATH, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

`data/` 配下にParquetファイル、`metadata/` 配下にavro・jsonファイルが入っています。  
Icebergの実データはParquetで保存され、メタデータはjsonやavroで構成されます。

## 4. タイムトラベルを体験してみよう

タイムトラベルを体験するため、テーブルにいくつかの変更を加えてみましょう。

In [ ]:
# テーブル変更前のスナップショットIDを記録
first_snapshot_id = table.current_snapshot().snapshot_id
print(f"最初のスナップショットID: {first_snapshot_id}")

### passenger_countを全部5にしてみる

In [ ]:
# 変更前
arrow = table.scan().to_arrow()
print("変更前のpassenger_count:")
conn.execute("SELECT passenger_count FROM arrow LIMIT 3").df()

In [ ]:
# passenger_countを全部5にしてみる
import pyarrow.compute as pc

all_data = table.scan().to_arrow()
updated = all_data.set_column(
    all_data.schema.get_field_index("passenger_count"),
    "passenger_count",
    pa.array([5] * len(all_data), type=pa.int64())
)
table.overwrite(updated)

arrow = table.scan().to_arrow()
print("変更後のpassenger_count:")
conn.execute("SELECT passenger_count FROM arrow LIMIT 3").df()

### 全レコードを削除！！！

In [ ]:
# 変更前のレコード数
arrow = table.scan().to_arrow()
print(f"削除前のレコード数: {len(arrow)}")

In [ ]:
# 全レコードを削除
empty = table.scan().to_arrow().slice(0, 0)  # 空のArrowテーブル
table.overwrite(empty)

arrow = table.scan().to_arrow()
print(f"削除後のレコード数: {len(arrow)}")

### タイムトラベル！！！

やれやれ...なんだかハチャメチャになってしまいましたね。。  
でも安心してください、Icebergには強力なタイムトラベルの仕組みがあります。

メタデータテーブル`.snapshots`にはtaxisテーブルへの変更のSNAPSHOTが記録されています。

In [ ]:
# スナップショット一覧を確認
snapshots = [
    {
        "snapshot_id": s.snapshot_id,
        "committed_at": s.timestamp_ms,
        "operation": s.summary.get("operation", "unknown") if s.summary else "unknown",
    }
    for s in table.snapshots()
]
pd.DataFrame(snapshots)

In [ ]:
# タイムトラベル: 最初のスナップショット時点のデータを参照
past_data = table.scan(snapshot_id=first_snapshot_id).to_arrow()
print(f"最初のスナップショット時点のレコード数: {len(past_data)}")
conn.execute("SELECT * FROM past_data LIMIT 5").df()

スナップショットIDを指定することで、過去の任意の断面にクエリできます。  
Icebergではデータを削除しても元のファイルは残っており、スナップショットを通じてアクセスできます。

In [ ]:
# ロールバック: 最初のスナップショットのデータを読み取り、テーブルを上書き
first_snapshot_data = table.scan(snapshot_id=first_snapshot_id).to_arrow()
table.overwrite(first_snapshot_data)

arrow = table.scan().to_arrow()
print(f"ロールバック後のレコード数: {len(arrow)}")
conn.execute("SELECT passenger_count FROM arrow LIMIT 3").df()

## 5. 同時実行制御を体験してみよう

IcebergはACIDをサポートしており、複数のクライアントが同時書き込みを行った場合でもテーブルの整合性を確保できます。  

2つのスレッドが同時にfare_amountを更新する状況を擬似的に再現してみます。

In [ ]:
import threading
from pyiceberg.exceptions import CommitFailedException

results = []

def update_fare(user_id, delta):
    try:
        data = table.scan().to_arrow()
        idx = data.schema.get_field_index("fare_amount")
        updated_col = pc.add(data.column("fare_amount"), delta)
        updated = data.set_column(idx, "fare_amount", updated_col)
        table.overwrite(updated)
        results.append(f"User {user_id}: fare_amount を {delta:+.1f} 変更しました")
    except CommitFailedException as e:
        results.append(f"User {user_id}: 競合を検出しました → {e}")
    except Exception as e:
        results.append(f"User {user_id}: エラー → {e}")

threads = [
    threading.Thread(target=update_fare, args=(1,  5.0)),  # ユーザー1: 5ドル増やす
    threading.Thread(target=update_fare, args=(2, -3.0)),  # ユーザー2: 3ドル減らす
]

for t in threads:
    t.start()
for t in threads:
    t.join()

for r in results:
    print(r)

片方のスレッドで `CommitFailedException` が発生したはずです。  
これはIcebergの楽観的並行性制御が競合を検出してコミットをabortさせたことを意味します。  
このように、Icebergは複数のクライアントが同時に書き込みを行った場合でもデータの整合性を担保できます。

Icebergの同時実行制御について詳しく知りたい方は以下を参考にしてください。  
- [How Apache Iceberg enables ACID compliance for data lakes](https://medium.com/snowflake/how-apache-iceberg-enables-acid-compliance-for-data-lakes-9069ae783b60)

## 6. スキーマエボリューションを体験してみよう

従来のHive Styleフォーマットでは、一部のスキーマ変更時にテーブル全体を作り直さなければならないことがありました。  
Icebergでは、テーブルを作り直すことなく[様々なスキーマ変更](https://iceberg.apache.org/docs/latest/evolution/)を適用し、新旧データを一貫して処理できます。

In [ ]:
# 現状のスキーマを確認
print("変更前のスキーマ:")
for field in table.schema().fields:
    print(f"  {field.field_id}: {field.name} ({field.field_type})")

In [ ]:
# fare_amount → fare にリネーム
with table.update_schema() as update:
    update.rename_column("fare_amount", "fare")

# trip_distance → distance にリネーム
with table.update_schema() as update:
    update.rename_column("trip_distance", "distance")

print("リネーム後のスキーマ:")
for field in table.schema().fields:
    print(f"  {field.field_id}: {field.name} ({field.field_type})")

In [ ]:
# 新しいカラム fare_per_distance_unit を追加
with table.update_schema() as update:
    update.add_column("fare_per_distance_unit", DoubleType(), doc="fare per distance unit")

print("カラム追加後のスキーマ:")
for field in table.schema().fields:
    print(f"  {field.field_id}: {field.name} ({field.field_type})")

In [ ]:
# fare / distance を fare_per_distance_unit に設定
data = table.scan().to_arrow()
fare_per_dist = pc.divide(
    data.column("fare").cast(pa.float64()),
    data.column("distance").cast(pa.float64())
)
idx = data.schema.get_field_index("fare_per_distance_unit")
updated = data.set_column(idx, "fare_per_distance_unit", fare_per_dist)
table.overwrite(updated)

result = table.scan(
    selected_fields=("vendor_id", "pickup_datetime", "fare", "distance", "fare_per_distance_unit")
).to_arrow()
conn.execute("SELECT * FROM result LIMIT 5").df()

このように、テーブルを作り直すことなく様々なスキーマ変更が適用できます。

さらに、スキーマ変更後でも過去のスナップショット(スキーマ変更前のデータ)にクエリできます。

In [ ]:
# スキーマ変更前のスナップショットを参照(fare_amountカラムが存在)
past = table.scan(snapshot_id=first_snapshot_id).to_arrow()
print(f"過去スナップショットのカラム: {past.schema.names}")
conn.execute("SELECT vendor_id, fare_amount, trip_distance FROM past LIMIT 5").df()